# Claude Code on the Cluster
### Operating an AI coding agent as research infrastructure — headless, reproducible, cost-aware

**Claude Code** is a coding assistant that runs in your terminal: it reads and edits your files and runs your
commands, but only inside a permission system *you* control. This workshop teaches you to drive it **without the
chat UI** — from scripts and notebooks — so you can automate real work on a shared cluster and see exactly what
it did and what it cost.

> **The mental model (30 seconds).**
> - **The harness** is the `claude` program you install and run in your terminal — *this is the thing the
>   workshop configures.* (The **model** is the AI it talks to over the network; we don't touch that.)
> - It works in an **agentic loop**: read a file → run a command → look at the result → act again, until done.
> - **Tools** are the concrete actions it can take: `Read`, `Edit`, `Bash`.
> - **Permission mode** is your autonomy dial, from "read-only" to "edit files freely." *You* set it; the model
>   can't override it.
>
> Everything below is a setting or flag on that `claude` program.

**No prior workshop required — if you can run a Python cell, you're ready.** *(Did the build-it-yourself
workshops? §19 maps each piece you built by hand to its Claude Code counterpart.)*

**By the end you'll be able to** (1) drive `claude -p` headlessly and read its JSON output; (2) get
schema-validated structured output; (3) govern what the agent may do with permissions, hooks, and subagents;
(4) compose it into an audited, cluster-scale bug-fix — and (5) do all of it from Python via the Agent SDK.

**Part I — driving the CLI (headless):**

| # | Section | You will run… |
|---|---|---|
| 0–1 | Install & setup | version check + a zero-cost readiness probe |
| 2 | First headless call | `claude -p … --output-format json` |
| 3 | Structured output | `--json-schema` |
| 4–5 | Project memory & slash commands | `CLAUDE.md`, a custom `/command` |
| 6–7 | Permissions & hooks | same prompt *denied* then *allowed*; a blocking hook |
| 8–10 | Subagents, sessions, MCP | delegation, `--resume`/`--fork-session`, your own tool server |
| 11 | Skills | the model reaches for a skill *on its own* |
| 12–13 | Capstone & scale-out | an agent fixes a failing test; a costed fan-out |

*Part II (§14–§18) does all of this from Python via the Agent SDK.*

**Before you start:** `claude --version` → v2.x and authenticated (§0 has the no-sudo install + login); network
egress to `api.anthropic.com` (on Midway, `test`/`caslake` nodes and login nodes have it); the `AI` conda env
and `cc_utils.py` beside this notebook. **Budget: ≈ 30 short `haiku` calls, well under $1.** Every check below is
anchored on something **deterministic** — an exit code, a file that does or doesn't exist, or a token the model
couldn't have invented. We never grade the *wording* of a reply.

## 0 · Install & authenticate (once) — then check you're ready

If `claude --version` already prints a **2.x** build, skip to §1. Otherwise the cell below has the one-time
install and login, **commented out** so it's a safe no-op — copy the lines into a terminal. None of it needs
`sudo`. The last lines *do* run: a zero-cost readiness check.

In [1]:
# ── ONE-TIME SETUP — run these in a TERMINAL (commented, so running this cell does nothing). ──
#
# 1) Install the CLI (no sudo) — pick ONE:
#      curl -fsSL https://claude.ai/install.sh | bash          # native installer -> ~/.local/bin
#      npm config set prefix ~/.npm-global && npm install -g @anthropic-ai/claude-code   # or via npm (Node >= 18)
#
# 2) Log in ONCE — pick ONE:
#      claude setup-token                    # Pro/Max subscription (works headless over SSH)
#      export ANTHROPIC_API_KEY=sk-ant-...   # or a pay-per-token API key (from a chmod-600 file; never hard-code)
#
# 3) For Part II (§14–§18), the Python Agent SDK:
#      pip install claude-agent-sdk
#
# ── This part runs: a readiness check (no network, no cost). ──
import shutil
print("claude CLI :", "found — run `claude --version` to confirm it's v2.x"
      if shutil.which("claude") else "NOT FOUND — see the commented install lines above")
try:
    import claude_agent_sdk as _probe
    print("Agent SDK  :", _probe.__version__, "— Part II will run live")
except ImportError:
    print("Agent SDK  : not installed — Part II shows its code and skips the live call (pip install claude-agent-sdk)")

claude CLI : found — run `claude --version` to confirm it's v2.x


Agent SDK  : 0.2.110 — Part II will run live


## 1 · Setup — fail fast before anything expensive

We drive the real `claude` command as a subprocess. Three habits (baked into `cc_utils`) make that safe inside a
notebook: `stdin=DEVNULL` so a call can't hang waiting for input, a wall-clock `timeout` so a runaway agent is
bounded, and an isolated working directory so demos never touch your files. We check readiness in **three small
steps** so that if something breaks, you know exactly which thing.

In [2]:
# (1a) Import helpers and confirm the CLI is v2.x.
import subprocess, re, os, json, hashlib, shlex
import cc_utils as cc

version = subprocess.run(["claude", "--version"], capture_output=True, text=True).stdout.strip()
print("claude --version:", version)
assert re.search(r"\b2\.\d+\.\d+\b", version), "This workshop targets Claude Code v2.x"

claude --version: 2.1.201 (Claude Code)


In [3]:
# (1b) One throwaway workspace for every demo (removed in §19). cc.redact() hides machine paths.
ws = cc.Workspace()
print("scratch workspace:", cc.redact(ws.root), "(throwaway)")
print("config dir       : $CLAUDE_CONFIG_DIR (default ~/.claude) — your login + transcripts live here; we only read it")

scratch workspace: /project/<you>/.cache/tmp/ccws_k8p6etb2 (throwaway)
config dir       : $CLAUDE_CONFIG_DIR (default ~/.claude) — your login + transcripts live here; we only read it


In [4]:
# (1c) One cheap live call. If THIS fails, it's auth or network — fix that before going on.
preflight = cc.ask("Reply with exactly: PREFLIGHT-OK", ws.dir("preflight"), timeout=60)
print("is_error:", preflight["is_error"], "| reply:", preflight["result"].strip())
assert preflight["is_error"] is False, "Preflight failed — check `claude` auth / network egress."
print("setup OK —", cc.LEDGER.line())

is_error: False | reply: PREFLIGHT-OK
setup OK — cumulative: $0.0156 over 1 claude calls


## 2 · Your first headless call — see the *real* command

`claude -p "<prompt>"` runs **one non-interactive turn** and exits. Add `--output-format json` and you get a
machine-readable record instead of prose — the foundation of every pipeline here. First, the **literal command**,
exactly what you'd type in a terminal, so you see it before we wrap it:

In [5]:
# The exact argv. (stdin=DEVNULL + timeout are the safety habits from §1.)
cmd = ["claude", "-p", "What is 17 * 23? Reply with only the number.",
       "--output-format", "json", "--model", "haiku"]
print("$", shlex.join(cmd), "\n")

proc = subprocess.run(cmd, cwd=ws.dir("first"), stdin=subprocess.DEVNULL,
                      capture_output=True, text=True, timeout=60)
raw = json.loads(proc.stdout)
print("result     :", raw["result"])
print("cost (usd) :", raw["total_cost_usd"])
print("session_id :", raw["session_id"], "  # a UUID — you can --resume it later (§9)")
# What happened: one `claude` run returned a JSON record — the answer, its cost, and a session id.

$ claude -p 'What is 17 * 23? Reply with only the number.' --output-format json --model haiku 



result     : 391
cost (usd) : 0.0162893
session_id : 8f4bf07f-dc8d-4420-8623-88e7d55c0256   # a UUID — you can --resume it later (§9)


The keys you'll actually use: **`result`** (the text answer), **`total_cost_usd`**, **`session_id`**, and
**`is_error`**.

For the rest of the notebook we call **`cc.ask()`** instead — a thin wrapper around *that exact command* that
also adds the cost to a running ledger. Same call, one extra feature. (`cc.show()` just pretty-prints the result.
Peek at `cc_utils.py` any time — it's short.)

In [6]:
first = cc.ask("What is 17 * 23? Reply with only the number.", ws.dir("first"))
cc.show(first, label="first call (via cc.ask)")

print("\nanswer contains 391:", "391" in first["result"])   # deterministic check — we never grade the wording

┌─ first call (via cc.ask) ─────────────────────────────────────
│ 391
├─ telemetry ─────────────────────────────────────────────────
│ cost=$0.00349  turns=1  out_tokens=88  is_error=False
└─────────────────────────────────────────────────────────────

answer contains 391: True


**🎯 Try this.** Change `--model haiku` to `--model sonnet` in the raw `cmd` above and re-run it — watch
`total_cost_usd` change. Knowing the price of a call is what keeps a batch job's bill predictable.

## 3 · Structured output on tap — `--json-schema`

Research work constantly needs *structured extraction*: pull fields out of an abstract, classify a passage, emit
a row for a table. Instead of parsing free text with regexes, hand `claude` a **JSON Schema** and it returns an
object that fits it. Below, `echo=True` prints the real command so you see the new `--json-schema` flag:

In [7]:
schema = json.dumps({
    "type": "object",
    "properties": {
        "organism": {"type": "string"},
        "sample_size": {"type": "integer"},
        "significant": {"type": "boolean"},
    },
    "required": ["organism", "sample_size", "significant"],
})
abstract = ("We treated 48 zebrafish embryos with compound X and observed a statistically "
            "significant increase in fluorescence (p < 0.01).")

extracted = cc.ask("Extract the fields in the schema from this abstract: " + abstract,
                   ws.dir("schema"), extra=["--json-schema", schema], echo=True)

# --json-schema returns the object as a JSON string in "result", so we parse it once:
record = json.loads(extracted["result"]) if isinstance(extracted["result"], str) else extracted["result"]
print("\ntyped record:", record)
print("all fields present:", {"organism", "sample_size", "significant"} <= set(record))

→ claude -p 'Extract the fields in the schema from this abstract: We treated 48 zebrafish embryos with compound X and observed a statistically significant increase in fluorescence (p < 0.01).' --model haiku --output-format json --json-schema '{"type": "object", "properties": {"organism": {"type": "string"}, "sample_size": {"type": "integer"}, "significant": {"type": "boolean"}}, "required": ["organism", "sample_size", "significant"]}'



typed record: {'organism': 'zebrafish', 'sample_size': 48, 'significant': True}
all fields present: True


**🎯 Try this.** Add a field to the schema — e.g. `"effect_direction"` (a string, `"increase"` or
`"decrease"`) — re-run, and watch Claude fill it in. Schemas are how you make an LLM's output *safe to feed to
the next stage of a pipeline*.

## 4 · Teaching Claude your project — `CLAUDE.md`

`CLAUDE.md` is a plain Markdown file that Claude Code **auto-loads** whenever a session starts in that directory.
It's where you write what a new lab member would need: the env-activation line, where data lives, the test
command, "never write to the raw-data folder." Put something here if Claude should know it in *every* future
session on this project.

We prove the mechanism two ways: a folder whose `CLAUDE.md` plants a secret codename (Claude reports it), and a
sibling folder with **no** `CLAUDE.md` (Claude can't produce a token it never saw). That second case — a
**negative control** — is the strongest evidence: an 8-character random token can't be guessed.

In [8]:
with_mem = ws.dir("mem_yes")
ws.write("mem_yes/CLAUDE.md",
         "# Lab conventions\n\nWhen asked for the project codename, reply with exactly: ZEPHYR-9\n")
no_mem = ws.dir("mem_no")     # deliberately NO CLAUDE.md

answer_yes = cc.ask("What is the project codename?", with_mem)
answer_no  = cc.ask("What is the project codename?", no_mem)
print("with CLAUDE.md    -> says ZEPHYR-9:", "ZEPHYR-9" in answer_yes["result"].upper())
print("without CLAUDE.md -> says ZEPHYR-9:", "ZEPHYR-9" in answer_no["result"].upper(), "(negative control)")

with CLAUDE.md    -> says ZEPHYR-9: True
without CLAUDE.md -> says ZEPHYR-9: False (negative control)


**🎯 Try this.** The workshop ships a realistic template at `sample-lab/CLAUDE.md`. Its core idea, in three
lines you can drop into any repo:

```markdown
## Conventions
- Run the tests with: cd analysis && python -m pytest -q
- Raw data in /project/<pi>/data is READ-ONLY — never write or delete there.
```

In the interactive lab you run `/init` to *generate* one of these for your own repo automatically.

## 5 · Reusable prompts — slash commands

A slash command is a **saved, versioned prompt** stored in `.claude/commands/<name>.md`. Check it into git and
your whole lab shares the same `/add-test`, `/repro`, `/slurm-doctor`. Running `/greet` is literally
`claude -p "/greet"` — the command file supplies the prompt. `setting_sources="project"` tells the CLI to load
*this folder's* `.claude/` (and not your personal settings), keeping the demo self-contained.

In [9]:
ws.write("slash/.claude/commands/greet.md",
         "---\ndescription: Workshop greeting\n---\n"
         "Respond with exactly this line and nothing else: GREETING-TOKEN-7QX4\n")

reply = cc.ask("/greet", ws.dir("slash"), setting_sources="project")
print("reply:", reply["result"].strip())
print("the /greet command ran:", "GREETING-TOKEN-7QX4" in reply["result"])

reply: GREETING-TOKEN-7QX4
the /greet command ran: True


**🎯 Try this.** `sample-lab/.claude/commands/add-test.md` is a *parameterised* command — `/add-test <function>`
— that also restricts its own tools via frontmatter:

```markdown
---
description: Write a pytest test for a function the user names
allowed-tools: Read, Edit(analysis/**)
---
Write a focused pytest test for `$ARGUMENTS` in analysis/. Cover a normal case and one edge case.
```

Try writing a `/summarize` command that takes a filename in `$ARGUMENTS` and returns a three-bullet summary.

## 6 · Who's in charge — the permission system

This is the section that earns an agent a place on a shared cluster: **what the agent may *do* is decided by the
permission system, not by the model's good intentions.** We send the *same* prompt twice, changing only the
flags. Default: the `Write` tool is denied, so the file never appears. With `--allowedTools Write`: the same
request succeeds.

> **Who wins when rules conflict?** Precedence is **`deny` rule → `PreToolUse` hook → `allow` rule → default
> (deny)**. A `deny` always wins; a tool nobody allowed is denied. (We rely on this in §7 and §17.)

In [10]:
PROMPT = ("You must use the Write tool to create a file named out.txt in the current "
          "directory containing exactly: HELLO-FROM-CLAUDE")

deny_dir = ws.dir("perm_deny")
denied = cc.ask(PROMPT, deny_dir)                                 # default mode: Write is not allowed
denied_file = os.path.exists(os.path.join(deny_dir, "out.txt"))

allow_dir = ws.dir("perm_allow")
allowed = cc.ask(PROMPT, allow_dir, allowed=["Write"], echo=True)  # explicitly allow Write
allow_path = os.path.join(allow_dir, "out.txt")

print("\nDEFAULT : file created?", denied_file,
      "| denials:", [d.get("tool_name") for d in denied.get("permission_denials", [])])
print("ALLOWED : file created?", os.path.exists(allow_path),
      "| contents:", open(allow_path).read().strip() if os.path.exists(allow_path) else None)
print("\n-> identical prompt, opposite outcome: the permission system decided, not the model.")

→ claude -p 'You must use the Write tool to create a file named out.txt in the current directory containing exactly: HELLO-FROM-CLAUDE' --model haiku --output-format json --allowedTools Write



DEFAULT : file created? False | denials: ['Write']
ALLOWED : file created? True | contents: HELLO-FROM-CLAUDE

-> identical prompt, opposite outcome: the permission system decided, not the model.


**🎯 Try this.** `sample-lab/.claude/settings.json` allows editing `analysis/**` but denies editing the test file:

```json
{"permissions": {"allow": ["Edit(analysis/**)"], "deny": ["Edit(analysis/test_stats.py)"]}}
```

Predict: if you ask Claude to "fix the failing test by editing the test file", what happens? *(It can't —
`deny` beats `allow`, always. A deny is a hard veto.)*

## 7 · Hooks — policy the model can't forget

`CLAUDE.md` is *advice*; a **hook** is *policy*. A hook is a shell command the harness runs on a lifecycle event
(`PreToolUse`, `PostToolUse`, `SubagentStop`, …), set in `.claude/settings.json`, and it runs **no matter what
the model decides**. Two patterns cover most needs: a `PreToolUse` hook that **blocks** a tool, and a
`PostToolUse` hook that **audits** one. (Each is its own cell below.)

In [11]:
# (7a) A PreToolUse hook that BLOCKS every Write/Edit.
#   exit 2 = tell Claude "no, stop this tool call" (a non-zero exit blocks it)
#   >&2    = write the reason to the error channel so Claude sees it
block = ws.write("hook_block/block.sh", "#!/bin/bash\necho 'policy: writes forbidden here' >&2\nexit 2\n")
os.chmod(block, 0o755)                                   # 0o755 = make the script executable
ws.write("hook_block/.claude/settings.json", json.dumps({"hooks": {"PreToolUse": [
    {"matcher": "Write|Edit", "hooks": [{"type": "command", "command": block}]}]}}))

block_dir = ws.dir("hook_block")
cc.ask("Use the Write tool to create protected.txt containing HELLO.",
       block_dir, allowed=["Write"], setting_sources="project")
print("PreToolUse block -> protected.txt exists?",
      os.path.exists(os.path.join(block_dir, "protected.txt")), "(False = the hook blocked it)")

PreToolUse block -> protected.txt exists? False (False = the hook blocked it)


In [12]:
# (7b) A PostToolUse hook that APPENDS an audit line after each successful Write.
audit_log = os.path.join(ws.dir("hook_audit"), "audit.log")
post = ws.write("hook_audit/post.sh",
                "#!/bin/bash\necho \"$(date -u) claude used a tool\" >> " + audit_log + "\nexit 0\n")
os.chmod(post, 0o755)
ws.write("hook_audit/.claude/settings.json", json.dumps({"hooks": {"PostToolUse": [
    {"matcher": "Write", "hooks": [{"type": "command", "command": post}]}]}}))

cc.ask("Use the Write tool to create note.txt containing HI.",
       ws.dir("hook_audit"), allowed=["Write"], setting_sources="project")
print("PostToolUse audit -> log line written?", os.path.exists(audit_log))
if os.path.exists(audit_log):
    print("   log says:", open(audit_log).read().strip())

PostToolUse audit -> log line written? True
   log says: Sat Jul  4 00:02:24 UTC 2026 claude used a tool


**🎯 Try this.** A hook turns "please don't touch the raw data" from a polite `CLAUDE.md` note into a rule the
agent *cannot* break. Which event would you use to inject a reminder into *every* prompt automatically? *(Hint:
`UserPromptSubmit`.)*

## 8 · Subagents — hand a job to a scoped worker

A **subagent** (`.claude/agents/<name>.md`) is a helper the main agent can delegate to. It has its own
instructions, its own context, and — crucially — its own **tool allowlist**: a `reviewer` limited to
`Read, Grep` *cannot* modify files even if asked. Delegation is the *model's* choice, so we don't trust the
transcript — we prove it with a `SubagentStop` hook that writes a marker file only when a subagent finishes.

In [13]:
sub_dir = ws.dir("subagent")
ws.write("subagent/main.py", "def add(a, b):\n    return a - b   # looks suspicious\n")
ws.write("subagent/.claude/agents/reviewer.md",
         "---\nname: reviewer\ndescription: Reviews code files. Use whenever asked to review code.\n"
         "tools: Read, Grep\n---\nYou are a meticulous code reviewer. Report the bug you find.\n")
marker = os.path.join(sub_dir, "subagent.log")
sh = ws.write("subagent/sa.sh", "#!/bin/bash\necho ran >> " + marker + "\nexit 0\n")
os.chmod(sh, 0o755)
ws.write("subagent/.claude/settings.json", json.dumps({"hooks": {"SubagentStop": [
    {"hooks": [{"type": "command", "command": sh}]}]}}))

summary = cc.ask("Use the reviewer subagent to review main.py. You must delegate to the reviewer via the "
                 "Task tool, then report what it found.",
                 sub_dir, allowed=["Task"], setting_sources="project", timeout=120)
print("delegation happened (SubagentStop fired):", os.path.exists(marker))
print("main agent's summary:", summary["result"].strip()[:110], "...")

delegation happened (SubagentStop fired): True
main agent's summary: ## Review Results for main.py

The reviewer agent found **one critical correctness bug**:

**Issue: Logic Erro ...


**🎯 Try this.** A read-only critic (`Read, Grep`) is *safe to run in a loop* — it can't break anything. Where
in your workflow would one help? *(A security reviewer, a docstring checker, a "did this change the API?"
checker.)*

## 9 · Sessions — resume and branch a conversation

Every headless call is a **session** with a transcript saved on disk. `--resume <session_id>` continues it with
full memory (the basis of resumable pipelines); `--fork-session` branches it into a *new* id, so you can explore
two continuations without them interfering. We plant a token, then (a) find the transcript, (b) resume and
recall it, (c) show a fresh session *can't* recall it (negative control), and (d) fork to a new id.

In [14]:
session_dir = ws.dir("session")
turn1 = cc.ask("Remember this token for later: MAGENTA-FALCON-42. Just reply: noted.", session_dir)
session_id = turn1["session_id"]

transcript = cc.find_transcript(session_id)
print("(a) transcript on disk:", bool(transcript),
      "| token in it:", bool(transcript) and "MAGENTA-FALCON-42" in open(transcript).read())

resumed = cc.ask("Reply with only the token I asked you to remember.", session_dir,
                 extra=["--resume", session_id], echo=True)
print("(b) --resume recalls it     :", "MAGENTA-FALCON-42" in resumed["result"].upper())

fresh = cc.ask("Reply with only the token I asked you to remember, if any.", ws.dir("session_fresh"))
print("(c) fresh session recalls it:", "MAGENTA-FALCON-42" in fresh["result"].upper(), "(negative control)")

forked = cc.ask("Say hi.", session_dir, extra=["--resume", session_id, "--fork-session"])
print("(d) --fork-session new id   :", forked["session_id"] != session_id,
      "|", session_id[:8], "->", forked["session_id"][:8])

(a) transcript on disk: True | token in it: True
→ claude -p 'Reply with only the token I asked you to remember.' --model haiku --output-format json --resume 7306a666-6b56-4a63-8406-57fb9225d610


(b) --resume recalls it     : True


(c) fresh session recalls it: False (negative control)


(d) --fork-session new id   : True | 7306a666 -> f034d8c6


**🎯 Try this.** Pass `--session-id <fixed-uuid>` to *name* a pipeline's session so you can always find and
replay it. Resuming beats starting fresh whenever later steps depend on what the agent already discovered — a
multi-step analysis, a long refactor.

## 10 · MCP — plug in your own tools

The **Model Context Protocol** lets Claude Code call *your* tools: a database, a `squeue` wrapper, an internal
API. You run a small server; its tools show up as `mcp__<server>__<tool>` (namespaced away from the built-ins).
This is how Claude Code becomes cluster-native. We start the dependency-free server in `sample-lab/mcp_server.py`
(it exposes a `storage_quota` tool). Two flags matter headless: MCP tools are **deny-by-default**, so we allow
`mcp__lab__*`; and `--strict-mcp-config` makes the demo ignore any other MCP config on your system:

```
claude -p "..." --allowedTools mcp__lab__storage_quota --strict-mcp-config \
    --mcp-config '{"mcpServers":{"lab":{"command":"python3","args":["mcp_server.py"]}}}'
```

In [15]:
server = os.path.abspath("sample-lab/mcp_server.py")
mcp_config = json.dumps({"mcpServers": {"lab": {"command": "python3", "args": [server]}}})

mcp_reply = cc.ask("Call the storage_quota MCP tool and report exactly the text it returns.",
                   ws.dir("mcp"), allowed=["mcp__lab__storage_quota"],
                   extra=["--mcp-config", mcp_config, "--strict-mcp-config"], timeout=120)
cc.show(mcp_reply, label="MCP call")
print("the tool's result reached the reply:", "812 GB" in mcp_reply["result"] or "41%" in mcp_reply["result"])

┌─ MCP call ────────────────────────────────────────────────────
│ The storage_quota tool has returned its result. As reported above:
│ 
│ **You are using 812 GB of your 2 TB quota on /project (41%).**
├─ telemetry ─────────────────────────────────────────────────
│ cost=$0.07528  turns=2  out_tokens=225  is_error=False
└─────────────────────────────────────────────────────────────
the tool's result reached the reply: True


**🎯 Try this.** Open `sample-lab/mcp_server.py` — it has a `TODO`: add a `gpu_free` tool (no inputs) to `TOOLS`
and handle it in `call_tool()`, then re-run this cell asking Claude to call your new tool. Interactively you'd
register a server permanently with `claude mcp add`; here we pass it inline.

## 11 · Skills — capabilities the model reaches for *on its own*

A **skill** is a folder — `.claude/skills/<name>/SKILL.md` — holding a `name`, a one-line `description`, a body
of instructions, and (optionally) reference files and scripts. The magic is **progressive disclosure**: at
startup the agent reads *only* each skill's name + description (a short line each, not the full body); when your request matches a
description it pulls in that `SKILL.md` body; and it opens the bundled files *only if it needs them*. So you can
ship a whole library of expertise that costs almost no context until it is relevant.

The distinction to hold onto — a **slash command is a prompt _you_ fire; a skill is a capability the _model_
fires** when your ask matches its description — and unlike a command it can carry its own reference files and
scripts:

| Surface | Loaded… | Invoked by | Carries |
|---|---|---|---|
| `CLAUDE.md` (§4) | always, in full | — | project facts & conventions |
| slash command (§5) | on `/name` | **you**, explicitly | one reusable prompt (+ tool limits) |
| **skill (§11)** | name+desc always; body on match | **the model** when your ask fits its `description` (or you, via `/name`) | a procedure + optional scripts / reference files, pulled in on demand |
| subagent (§8) | on delegation | model or you | a scoped worker with its *own* context window |
| MCP (§10) | tool schema always | the model, as a tool call | external tools over a protocol |

**Anatomy of `SKILL.md`** — YAML frontmatter, then Markdown:
- `name:` lowercase-with-hyphens, ≤ 64 chars (not `claude`/`anthropic`).
- `description:` the load-bearing line — the model matches on it, so state **what it does _and_ when to use it**, in the third person, with trigger words (*"Use when the user asks about a pending or failed Slurm job…"*). ≤ 1024 chars.
- `allowed-tools:` *(optional, **CLI only**)* pin the skill to a safe tool set — e.g. `Read` for a read-only
  skill. *(The Agent SDK ignores this field; restrict tools there with the SDK's own `allowed_tools`.)*
- The body is instructions; keep it tight (< ~500 lines) and link heavy detail to sibling files it reads on demand.

Skills live in the **project** (`.claude/skills/`, shared via git), your **personal** `~/.claude/skills/`, or a
**plugin**. The next cell writes one into a scratch project and — without us ever typing its name — watches the
model decide to use it.


In [ ]:
# A skill is a FOLDER: SKILL.md (name + description + instructions) plus optional reference files.
# This one teaches the agent to explain a stuck Slurm job — and it is deliberately READ-ONLY.
sk = "skills/.claude/skills/slurm-triage"
ws.write(sk + "/SKILL.md",
    "---\n"
    "name: slurm-triage\n"
    "description: Explain why a Slurm job is pending, stuck, or failed by reading its scontrol or\n"
    "  squeue output. Use when the user asks about a queued, pending, stuck, or failed HPC or Slurm job.\n"
    "allowed-tools: Read\n"
    "---\n\n"
    "# Slurm job triage (read-only)\n\n"
    "1. Read the `scontrol show job` output the user names.\n"
    "2. Find `JobState` and, if pending, the `Reason=(...)` code.\n"
    "3. Look up the code in `reference.md` (this folder) and explain it in plain language.\n"
    "4. Recommend the *smallest safe* change. NEVER extend time limits, cancel jobs, or edit\n"
    "   allocations — those need privileges the user may not have.\n\n"
    "Begin every diagnosis with the tag `[slurm-triage]` on its own line.\n")
ws.write(sk + "/reference.md",                       # a bundled file the model opens ONLY if it needs it
    "# Slurm pending-reason codes\n"
    "- `(Resources)` waiting for free nodes; normal. Wait, or request less.\n"
    "- `(Priority)` lower priority than queued jobs. Wait; check fairshare.\n"
    "- `(QOSMaxCpuPerUserLimit)` you hit your per-user QOS CPU cap. Let jobs finish or ask for fewer CPUs.\n"
    "- `(MaxSubmitJobsPerAccount)` the shared account hit its job-count cap. Wait or coordinate.\n"
    "- `(ReqNodeNotAvail,...)` pinned to a down/drained node. Drop the -w/--nodelist constraint.\n")
ws.write("skills/job.txt",
    "JobId=4812203 JobName=train_gpt Account=pi-smith QOS=normal\n"
    "JobState=PENDING Reason=(QOSMaxCpuPerUserLimit)\n"
    "Partition=caslake NumNodes=4 NumCPUs=192 TimeLimit=1-00:00:00\n")

# We never type "/slurm-triage". We describe the problem; the model matches the skill's
# description, loads it, and opens reference.md on its own — progressive disclosure in action.
proc = cc.run_claude(
    "My Slurm job won't start — it's stuck PENDING. The scontrol output is in job.txt. "
    "Why is it pending, and what's the smallest safe thing I can do?",
    ws.dir("skills"),
    allowed=["Read", "Skill"],          # `Skill` is the tool the model uses to run a skill
    setting_sources="project",           # discover THIS project's .claude/skills/
    output="stream-json", extra=["--verbose"], timeout=150)

events = [json.loads(l) for l in proc.stdout.splitlines() if l.strip()]
calls  = [b for e in events if e.get("type") == "assistant"
          for b in e["message"].get("content", []) if b.get("type") == "tool_use"]
tools  = [b["name"] for b in calls]
reads  = [b.get("input", {}).get("file_path", "") for b in calls if b["name"] == "Read"]
result = next((e for e in reversed(events) if e.get("type") == "result"), {})
answer = result.get("result", "")
cc.LEDGER.add(result)

print("tool calls the model made:", tools)                        # e.g. ['Read', 'Skill', 'Read']
print("- invoked the skill unprompted   :", "Skill" in tools)
print("- opened reference.md on demand  :", any(p.endswith("reference.md") for p in reads))
print("- skill body loaded (tag present):", "[slurm-triage]" in answer)
print(cc.LEDGER.line())


**🎯 Try this.** The lab ships this skill — and an *unfinished* one — in `sample-lab/.claude/skills/`. Read the
working `slurm-triage/SKILL.md`, then finish `sbatch-lint/SKILL.md`: write its one-line `description` (that line
is what makes the model pick it) and its procedure, using the `reference.md` already sitting beside it. Start
`claude` in the lab and ask *"will this sbatch script run?"* — the model should reach for it without your
naming it, and if it doesn't, tighten the `description`'s trigger words (that's the whole game). You can always
invoke it explicitly, like a command: `/sbatch-lint`.

**Skills scale into a library.** One skill per task, plus a top-level *router* skill whose description says
"start here," is how a real RCC support-assistant is built — `slurm-jobs`, `cluster-status`, `disk-usage`,
`docs-lookup`, each its own folder with its own reference files. Reach for a skill when a section of your
`CLAUDE.md` grows from *facts* into a multi-step *procedure*.

**Ship only what a normal user can run.** A skill you hand to others must assume ordinary rights. "Extend a
running job's wall-time," say, needs operator privileges most people don't have — so it has no business in a
shared skill. `slurm-triage` is deliberately **read-only**: it explains and recommends, never mutates. Keep
privileged or destructive actions out of skills, and behind the permission system (§6–§7).

*(Skills work headless and through the Agent SDK too — but the SDK is **hermetic** (§14), so it discovers none
unless you pass `setting_sources=["project"]` and allow the `Skill` tool. One more SDK caveat: a skill's own
`allowed-tools` is **ignored** there, so pin tools with the SDK's `allowed_tools` / permission callback instead.)*


## 12 · Capstone — an agent fixes a failing test, and you audit every step

Now it all composes. We plant a realistic bug — an off-by-one in a sliding-window mean — with a failing
`pytest`, then hand the repo to Claude under **`--permission-mode acceptEdits`** (auto-approve edits) with a
scoped tool list, asking it to make the tests pass **without touching the test file**. We watch the tool-by-tool
trace, then **re-run `pytest` ourselves** — the exit code flipping `1 → 0` is the ground truth, not anything the
model claims. The lesson: *the permission mode, not the model, made autonomous editing possible.* (Three cells:
plant + baseline, run the agent, independently verify.)

In [16]:
# (12a) Plant the bug + a failing test suite, and confirm it starts red.
repo = ws.dir("capstone")
ws.write("capstone/window.py",
         "def moving_average(xs, k):\n"
         "    # k-point trailing moving average; should return len(xs)-k+1 values\n"
         "    out = []\n"
         "    for i in range(len(xs) - k):          # BUG: off by one; needs len(xs)-k+1\n"
         "        out.append(sum(xs[i:i+k]) / k)\n"
         "    return out\n")
ws.write("capstone/test_window.py",
         "from window import moving_average\n\n"
         "def test_len():\n    assert len(moving_average([1,2,3,4,5], 3)) == 3\n\n"
         "def test_values():\n    assert moving_average([1,2,3,4,5], 3) == [2.0, 3.0, 4.0]\n\n"
         "def test_window_of_one():\n    assert moving_average([4,5,6], 1) == [4.0, 5.0, 6.0]\n")
test_sha = hashlib.sha256(open(os.path.join(repo, "test_window.py")).read().encode()).hexdigest()

def pytest_exit(cwd):
    r = subprocess.run(["python", "-m", "pytest", "-q"], cwd=cwd, capture_output=True, text=True, timeout=120)
    return r.returncode

print("baseline pytest exit (nonzero = red):", pytest_exit(repo))

baseline pytest exit (nonzero = red): 1


In [17]:
# (12b) Hand the repo to the agent under acceptEdits and render the tool-by-tool trace.
proc = cc.run_claude(
    "Run the tests with 'python -m pytest -q'. One test fails due to a bug in window.py. Fix the bug in "
    "window.py, then re-run until all tests pass. Do NOT modify the test file.",
    repo, allowed=["Bash", "Read", "Edit", "Write"], permission_mode="acceptEdits",
    output="stream-json", extra=["--verbose", "--max-turns", "20", "--max-budget-usd", "0.50"], timeout=240)

print("--- tool-by-tool trace ---")
types, result_evt = cc.render_trace(proc.stdout)
cc.LEDGER.add(result_evt)

--- tool-by-tool trace ---
  step  tool     detail
  ----  -------  ----------------------------------------------
     1  Bash     python -m pytest -q
     2  Read     window.py
     3  Edit     window.py
     4  Bash     python -m pytest -q


{'type': 'result',
 'subtype': 'success',
 'is_error': False,
 'api_error_status': None,
 'duration_ms': 12176,
 'duration_api_ms': 12281,
 'ttft_ms': 1442,
 'ttft_stream_ms': 795,
 'time_to_request_ms': 43,
 'num_turns': 5,
 'result': 'All tests pass! The bug in window.py was an off-by-one error in the range. The loop needed to iterate one more time to produce the correct number of moving averages: `range(len(xs) - k + 1)` instead of `range(len(xs) - k)`.',
 'stop_reason': 'end_turn',
 'session_id': 'b41fc61c-5a38-4b9c-8ed6-8dd3a9879ce8',
 'total_cost_usd': 0.0373914,
 'usage': {'input_tokens': 42,
  'cache_creation_input_tokens': 9609,
  'cache_read_input_tokens': 121214,
  'output_tokens': 1072,
  'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0},
  'service_tier': 'standard',
  'cache_creation': {'ephemeral_1h_input_tokens': 9609,
   'ephemeral_5m_input_tokens': 0},
  'inference_geo': 'not_available',
  'iterations': [{'input_tokens': 8,
    'output_tokens': 13

In [18]:
# (12c) Independently re-run pytest and confirm the test file is untouched. THIS is the ground truth.
print("pytest exit now (0 = green):", pytest_exit(repo))
print("test file untouched (sha) :",
      hashlib.sha256(open(os.path.join(repo, "test_window.py")).read().encode()).hexdigest() == test_sha)
print("fix took %s turns, cost $%.4f" % (result_evt.get("num_turns"), result_evt.get("total_cost_usd") or 0))

pytest exit now (0 = green): 0
test file untouched (sha) : True
fix took 5 turns, cost $0.0374


**🎯 Take-home.** The real payoff is doing this on your *own* code. Copy `sample-lab/` to a git-clean scratch
dir, `cd` in, run `claude`, and try: *"run the tests, fix the failing one, don't touch the tests."* Then
`git diff` before you keep the change. The shipped `settings.json` denies editing the test file — try to make
Claude cheat and watch the deny rule stop it.

## 13 · From one call to a cluster pipeline

The `cc.ask()` you've used all along is the unit of a **batch job**: loop it over inputs and sum
`total_cost_usd`. Here's that fan-out over three tiny "log files"; the `run.sh` beside this notebook does the
same thing as a real Slurm job — an unattended agentic pipeline with the bill printed at the end.

In [19]:
fanout_dir = ws.dir("fanout")
logs = {
    "job1.err": "Traceback: torch.cuda.OutOfMemoryError: CUDA out of memory.",
    "job2.err": "srun: error: Unable to allocate resources: Requested partition config not available.",
    "job3.err": "Completed successfully in 00:12:44.",
}
print("classifying %d job logs:" % len(logs))
for name, text in logs.items():
    ws.write("fanout/" + name, text)
    verdict = cc.ask("In one word — OOM, PARTITION, TIMEOUT, or OK — classify the failure in %s." % name,
                     fanout_dir, allowed=["Read"], timeout=60)
    print("  %-9s -> %-10s ($%.4f)" % (name, verdict["result"].strip().split()[0][:10],
                                       verdict.get("total_cost_usd") or 0))
print("\n" + cc.LEDGER.line())

classifying 3 job logs:


  job1.err  -> OOM        ($0.0203)


  job2.err  -> PARTITION  ($0.0125)


  job3.err  -> OK         ($0.0125)

cumulative: $0.4228 over 20 claude calls


---
# Part II · Running Claude Code programmatically

Part I drove Claude Code the **first** way — shelling out to `claude -p` and parsing its JSON. For real
pipelines you often want to stay *inside* your program. There are three **doors** in, by how deeply you embed it:

| Door | What it is | Reach for it when… |
|---|---|---|
| **1 · CLI subprocess** | `claude -p … --output-format json`, from any language (all of §1–§13) | quick scripts, shell / Slurm glue |
| **2 · The Agent SDK** | `pip install claude-agent-sdk` — the *same engine*, as async Python (also TypeScript) | typed messages, in-process tools, Python permission / hook callbacks |
| **3 · Automation surfaces** | streaming stdin, GitHub Actions, the TS SDK | CI/CD, event-driven bots, long-lived sessions |

Part II covers **door 2** in depth (§14–§17), then surveys **door 3** (§18). *(These need `claude-agent-sdk`; if
it's missing they print their code and skip the live call, so the notebook still runs end-to-end.)*

**Part II map:** §14 one-shot `query()` · §15 multi-turn `ClaudeSDKClient` · §16 in-process `@tool` · §17 Python
permission/hook governance · §18 streaming stdin, GitHub Actions, TypeScript.

## 14 · The Agent SDK — Claude Code as a Python library

`pip install claude-agent-sdk` gives you the **same agent** that powers `claude -p`, as async Python. Instead of
parsing a JSON blob, you iterate a **stream of typed messages** and read attributes off them. Two entry points:
**`query()`** for a one-shot task, and **`ClaudeSDKClient`** for a multi-turn conversation (§15). Both take a
**`ClaudeAgentOptions`** — the typed twin of the CLI flags you already know:

| `ClaudeAgentOptions` | mirrors the CLI flag |
|---|---|
| `model="haiku"` | `--model` |
| `allowed_tools=[…]` / `disallowed_tools=[…]` | `--allowedTools` / `--disallowedTools` |
| `permission_mode="acceptEdits"` | `--permission-mode` |
| `cwd=…` | working directory |
| `mcp_servers={…}` | `--mcp-config` (+ in-process servers, §16) |
| `can_use_tool=fn`, `hooks={…}` | *(SDK-only)* Python governance (§17) |
| `max_turns`, `resume`, `session_id` | `--max-turns`, `--resume`, `--session-id` |

**One gotcha vs. the CLI:** `setting_sources` defaults to `None`, so the SDK is **hermetic** — it loads no
`CLAUDE.md` or filesystem settings unless you pass e.g. `setting_sources=["project"]`. The next cell imports the
SDK (guarded) and defines `sdk_collect` (final text + result message from a stream); we run async code with
`cc.run_async`. Then §14's demo — a one-shot `query()`, the twin of §2: the typed `ResultMessage` carries the
same `total_cost_usd` / `session_id` you parsed from JSON, with no `json.loads`.

In [20]:
# Guarded import: the notebook still runs top-to-bottom if the SDK is absent.
try:
    import claude_agent_sdk as sdk
    from claude_agent_sdk import (
        query, ClaudeSDKClient, ClaudeAgentOptions,
        AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,
        tool, create_sdk_mcp_server,
        PermissionResultAllow, PermissionResultDeny, HookMatcher,
    )
    HAVE_SDK = True

    def sdk_collect(messages):
        """From a list of SDK messages, return (assistant_final_text, ResultMessage)."""
        text, result = [], None
        for m in messages:
            if isinstance(m, AssistantMessage):
                text += [b.text for b in m.content if isinstance(b, TextBlock)]
            elif isinstance(m, ResultMessage):
                result = m
        return "\n".join(text).strip(), result

    print("claude-agent-sdk", sdk.__version__, "loaded — the SDK demos below will run live.")
except ImportError:
    HAVE_SDK = False
    print("claude-agent-sdk not installed — SDK cells will print their code and skip the live call.")
    print("Install with:  pip install claude-agent-sdk   (then restart the kernel)")

claude-agent-sdk 0.2.110 loaded — the SDK demos below will run live.


In [21]:
async def one_shot():
    opts = ClaudeAgentOptions(model="haiku", max_turns=1)
    msgs = [m async for m in query(prompt="What is 17 * 23? Reply with only the number.", options=opts)]
    return sdk_collect(msgs)

if HAVE_SDK:
    text, res = cc.run_async(one_shot())
    cc.LEDGER.add({"total_cost_usd": res.total_cost_usd if res else 0})
    print("answer          :", text)
    print("typed result -> cost=$%.5f  turns=%d  is_error=%s  out_tokens=%s"
          % (res.total_cost_usd or 0, res.num_turns, res.is_error, (res.usage or {}).get("output_tokens")))
    print("session_id      :", res.session_id, "| is a UUID:", cc.is_uuid(res.session_id))
    print("answer has 391  :", "391" in text)   # same deterministic check as §2, no json.loads needed
else:
    print("(claude-agent-sdk not installed — skipping live call; the code above is the whole pattern.)")

answer          : 391
typed result -> cost=$0.00841  turns=1  is_error=False  out_tokens=80
session_id      : 672541fb-d39c-4936-8072-384351ad6832 | is a UUID: True
answer has 391  : True


## 15 · Multi-turn conversations — `ClaudeSDKClient`

`query()` is one-and-done. For a conversation that builds state — a multi-step analysis, an interactive
refactor — use **`ClaudeSDKClient`** as an async context manager: `await client.query(...)` to send a turn,
iterate `client.receive_response()` to drain the reply, repeat. State persists across turns *inside the same
`async with` block* (the SDK's `--resume`), so turn 2 remembers turn 1 with no session id to carry by hand.

In [22]:
async def multi_turn():
    async with ClaudeSDKClient(options=ClaudeAgentOptions(model="haiku")) as client:
        await client.query("Remember the number 128. Just reply: ok.")
        async for _ in client.receive_response():
            pass                                              # drain turn 1
        await client.query("What number did I ask you to remember? Reply with only the number.")
        msgs = [m async for m in client.receive_response()]  # turn 2
    return sdk_collect(msgs)

if HAVE_SDK:
    text, res = cc.run_async(multi_turn())
    cc.LEDGER.add({"total_cost_usd": res.total_cost_usd if res else 0})
    print("turn-2 recall  :", repr(text))
    print("remembered 128 :", "128" in text, "(state carried across turns, no session id to thread)")
else:
    print("(claude-agent-sdk not installed — skipping live call; the code above is the whole pattern.)")

turn-2 recall  : '128'
remembered 128 : True (state carried across turns, no session id to thread)


## 16 · Your own tools, *in-process* — `@tool` + `create_sdk_mcp_server`

§10 connected an **external** MCP server (a separate `python3` subprocess over stdio). The SDK adds a lighter
path: write a tool as a plain async function with **`@tool`**, bundle a few with **`create_sdk_mcp_server`**, and
pass the result to `options.mcp_servers`. These run **in your own process** — no subprocess — so they can use
live Python objects (an open DB handle, a loaded DataFrame). Naming/permissions match §10: the tool is
`mcp__<server>__<tool>`, allowed by that name. Here is §10's `storage_quota`, now in-process:

In [23]:
async def custom_tool_demo():
    @tool("storage_quota", "Return the caller's cluster storage quota.", {})
    async def storage_quota(args):
        # A real tool would hit a DB/API; being in-process, it can use any Python object in scope.
        return {"content": [{"type": "text", "text": "Using 812 GB of 2 TB on /project (41%)."}]}

    server = create_sdk_mcp_server(name="lab", version="1.0.0", tools=[storage_quota])
    opts = ClaudeAgentOptions(
        model="haiku",
        mcp_servers={"lab": server},                       # an IN-PROCESS server (no subprocess)
        allowed_tools=["mcp__lab__storage_quota"],         # same mcp__server__tool naming as §10
    )
    msgs = [m async for m in query(
        prompt="Call the storage_quota tool and report exactly what it returns.", options=opts)]
    return sdk_collect(msgs)

if HAVE_SDK:
    text, res = cc.run_async(custom_tool_demo())
    cc.LEDGER.add({"total_cost_usd": res.total_cost_usd if res else 0})
    print("reply:", text[:160])
    print("in-process tool result reached the reply:", "812 GB" in text)
else:
    print("(claude-agent-sdk not installed — skipping live call; the code above is the whole pattern.)")

reply: The storage_quota tool returns:

**Using 812 GB of 2 TB on /project (41%).**
in-process tool result reached the reply: True


## 17 · Governance in pure Python — permission callback + hooks

§6–§7 governed the agent with CLI flags and `settings.json`. The SDK lets you express the *same* policy as
**Python functions**, where your logic can be anything (check a path, call a policy service, log to your own
audit sink):

- **`can_use_tool(tool_name, tool_input, context)`** — an async *permission callback* returning
  `PermissionResultAllow()` or `PermissionResultDeny(...)`. It decides any tool **not** already allow-listed
  (allow-listed tools are pre-approved and skip it). Needs streaming mode, so we run it via `ClaudeSDKClient`.
- **Hooks** — the SDK twin of §7: `hooks={"PreToolUse": [HookMatcher(matcher="Write", hooks=[fn])]}`. A
  `PreToolUse` hook returning `permissionDecision: "deny"` blocks the call **even if the tool is allow-listed**.

Below: a deny-writes callback (the model asks to Write, your Python says no, the file never appears).

In [24]:
import tempfile

# can_use_tool decides any tool NOT already allow-listed. Here it enforces a deny-writes policy
# and records every tool it is asked to approve.
async def deny_writes_run():
    d = tempfile.mkdtemp(prefix="sdkperm_")
    asked = []
    async def can_use_tool(tool_name, tool_input, context):
        asked.append(tool_name)
        if tool_name in ("Write", "Edit"):
            return PermissionResultDeny(message="writes denied by policy")   # your veto, in Python
        return PermissionResultAllow()
    opts = ClaudeAgentOptions(
        model="haiku", cwd=d, can_use_tool=can_use_tool,
        disallowed_tools=["Bash"],          # so Write is the only file-creation path the callback gates
        max_turns=4,
    )
    async with ClaudeSDKClient(options=opts) as client:   # can_use_tool requires streaming mode
        await client.query("You must use the Write tool to create out.txt containing HELLO.")
        msgs = [m async for m in client.receive_response()]
    _, res = sdk_collect(msgs)
    cc.LEDGER.add({"total_cost_usd": res.total_cost_usd if res else 0})
    return asked, os.path.exists(os.path.join(d, "out.txt"))

if HAVE_SDK:
    asked, made = cc.run_async(deny_writes_run())
    print("tools the callback was asked to approve:", asked)
    print("callback returned Deny for Write -> out.txt created?", made, "(False = your Python blocked it)")
    print("-> the decision was a function you wrote; return PermissionResultAllow() and the Write goes through.")
else:
    print("(claude-agent-sdk not installed — skipping live call; the code above is the whole pattern.)")

tools the callback was asked to approve: ['Write', 'Read', 'Write']
callback returned Deny for Write -> out.txt created? False (False = your Python blocked it)
-> the decision was a function you wrote; return PermissionResultAllow() and the Write goes through.


In [25]:
# A PreToolUse HOOK vetoes the tool *before* it runs — even though Write is allow-listed here
# (the SDK twin of §7's blocking hook). A returned permissionDecision "deny" is a hard veto.
async def hook_veto():
    d = tempfile.mkdtemp(prefix="sdkhook_")
    async def block_writes(input_data, tool_use_id, context):
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": "policy: writes forbidden here",
        }}
    opts = ClaudeAgentOptions(
        model="haiku", cwd=d, allowed_tools=["Write"],          # allow-listed, yet…
        hooks={"PreToolUse": [HookMatcher(matcher="Write", hooks=[block_writes])]},
        max_turns=4,
    )
    async with ClaudeSDKClient(options=opts) as client:
        await client.query("You must use the Write tool to create out.txt containing HELLO.")
        msgs = [m async for m in client.receive_response()]
    _, res = sdk_collect(msgs)
    cc.LEDGER.add({"total_cost_usd": res.total_cost_usd if res else 0})
    return os.path.exists(os.path.join(d, "out.txt"))

if HAVE_SDK:
    made = cc.run_async(hook_veto())
    print("PreToolUse hook -> out.txt created?", made, "(False = the hook vetoed the allow-listed Write)")
else:
    print("(claude-agent-sdk not installed — skipping live call; the code above is the whole pattern.)")

PreToolUse hook -> out.txt created? False (False = the hook vetoed the allow-listed Write)


## 18 · More automation surfaces (door 3)

Three more ways to run Claude Code — one you can run now, two you wire into infrastructure.

**(a) A long-lived session over `stdin` — `--input-format stream-json`.** Door 1 sent one prompt per process.
With `--input-format stream-json` you stream *many* user turns as newline-delimited JSON into a **single
persistent** `claude` process and read events back with `--output-format stream-json`. This is how you drive a
multi-turn agent from a language with no SDK. Below, two turns go to one process and turn 2 recalls a codeword
only turn 1 gave — proving the session held.

In [26]:
def user_msg(text):
    return json.dumps({"type": "user", "message": {"role": "user", "content": text}})

stdin = user_msg("Remember the codeword: PELICAN-7. Reply: ok.") + "\n" + \
        user_msg("What codeword did I give you? Reply with only the codeword.") + "\n"

proc = subprocess.run(
    ["claude", "-p", "--input-format", "stream-json", "--output-format", "stream-json",
     "--verbose", "--model", "haiku", "--max-turns", "4"],
    input=stdin, capture_output=True, text=True, timeout=120)
events  = [json.loads(l) for l in proc.stdout.splitlines() if l.strip()]
results = [e for e in events if e.get("type") == "result"]
final   = results[-1].get("result", "") if results else ""
for r in results:
    cc.LEDGER.add({"total_cost_usd": r.get("total_cost_usd")})
print("turns answered      :", len(results))
print("final reply         :", repr(final[:60]))
print("recalled PELICAN-7  :", "PELICAN-7" in final.upper(), "(one process held the whole conversation)")

turns answered      : 2
final reply         : 'PELICAN-7'
recalled PELICAN-7  : True (one process held the whole conversation)


**(b) GitHub Actions — `@claude` on issues & PRs.** Anthropic ships an official action,
[`anthropics/claude-code-action`](https://github.com/anthropics/claude-code-action). Add `ANTHROPIC_API_KEY` as
a repo secret, drop this workflow in `.github/workflows/`, and mentioning **`@claude`** in an issue or PR
comment runs the same agent, in CI:

```yaml
name: Claude Code
on:
  issue_comment:
    types: [created]
jobs:
  claude:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: anthropics/claude-code-action@v1
        with:
          anthropic_api_key: ${{ secrets.ANTHROPIC_API_KEY }}
```

**(c) The TypeScript / Node SDK — same engine, other language.** `npm install @anthropic-ai/claude-agent-sdk`;
options are camelCase:

```javascript
import { query } from "@anthropic-ai/claude-agent-sdk";
for await (const msg of query({ prompt: "What is 17 * 23?", options: { model: "haiku" } }))
  if (msg.type === "result") console.log(msg.result, "$" + msg.total_cost_usd);
```

**Which door?** Shell / Slurm glue or a non-Python language → door 1. A Python (or TS) app that wants typed
messages and code-defined policy → door 2. CI or event-driven automation → door 3.

## 19 · Wrap-up

You drove Claude Code **two ways** — the **CLI** (§1–§13) and the **Agent SDK** (§14–§17) — and in both, every
capability is **governed and observable**:

- **A call = a structured record with a known cost** (CLI JSON ↔ SDK `ResultMessage`).
- **`CLAUDE.md`, schemas, slash commands, and skills** make the agent fit your project.
- **Permissions, hooks, subagents** decide *and enforce* what it may do — as flags/`settings.json` **or** Python callbacks.
- **Sessions & MCP** resume/branch work and add your own tools (external **or** in-process).
- **The capstone** composed it into an audited autonomous fix, then fanned out across a cluster.

### Take-home lab (do this by hand)
The interactive REPL can't run headless. Copy `sample-lab/` to a git-clean scratch dir, `cd` in, run `claude`,
and try: `/init`; **plan mode**, then `Shift+Tab` to cycle permission modes; fix the failing test under
`acceptEdits`; review with `git diff`; then `/add-test`, `/mcp`, `/agents`, `/cost`, and **write a skill** (`.claude/skills/`) the model reaches for on its own. Everything in `sample-lab/`
is a template to lift into your own repo. For the programmatic path, `pip install claude-agent-sdk` and adapt the
§14–§17 cells.

### If you took the build-it-yourself workshops
How each piece you built by hand maps to what you configured here:

| Concept you built by hand | Its Claude Code counterpart |
|---|---|
| a tool registry + tool-call parser | built-in `Read`/`Edit`/`Bash` + an observable trace |
| a self-repair / retry loop | the agent fixing `pytest` under a permission mode (§12) |
| an input/output guardrail | permission modes + hooks — governing *actions* (§6–§7) |
| a team of cooperating agents | tool-scoped subagents (§8) |
| a constrained / JSON decoder | `--output-format json` + `--json-schema` (§2–§3) |
| a cost-aware router | `--model` + `total_cost_usd` accounting |
| prompt templates | `CLAUDE.md` + slash commands (§4–§5) |
| a library of task recipes the model dips into | model-invoked, progressively-disclosed **skills** (§11) |

**Docs:** [`code.claude.com/docs`](https://code.claude.com/docs) · **Agent SDK:**
[`docs.claude.com/en/api/agent-sdk/overview`](https://docs.claude.com/en/api/agent-sdk/overview)

In [27]:
# Housekeeping: remove the throwaway workspace.
ws.cleanup()
print("Workshop complete.")
print(cc.LEDGER.line())

Workshop complete.
cumulative: $0.5255 over 27 claude calls
